# Inspect a SWE-smith trajectory collection

This notebook reads the artifacts produced by the SWE-smith `collect → eval → analyse` cluster pipeline. It does not start model or evaluation jobs.

The default pilot contains 30 tasks across three collection shards, with four trajectories at each of `0.6` and `0.7`: 8 deterministic sample slots and 240 trajectories in total.

From the repository root, preview or submit the cache-first pilot chain with:

```bash
DRY_RUN=1 cluster/submit_swesmith_pilot_with_cache.sh
cluster/submit_swesmith_pilot_with_cache.sh
```

If the pilot images are already cached, submit only `collect → eval → analyse` with `cluster/submit_swesmith_pilot.sh`. PBS stdout and stderr are written to `$RUN_ROOT/cluster-logs/`.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import JSON, Markdown, display

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

scratch = Path(os.environ.get("DEBUG_DEPO_SCRATCH", repo_root / "scratch"))
run_name = os.environ.get("RUN_NAME", "swesmith-pilot")
run_root = Path(os.environ.get("RUN_ROOT", scratch / "runs" / run_name))
print(f"Repository: {repo_root}")
print(f"Run root:   {run_root}")
if not run_root.exists():
    print("Run directory is not present yet. Set RUN_NAME or RUN_ROOT, or submit the pilot pipeline.")


## Collection coverage

Each collection shard has one manifest and 8 sample folders. This view checks task counts, rollout counts, temperatures, and collection errors before evaluation.

In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text()) if path.is_file() else {}

manifest_paths = sorted(run_root.glob("collection/shard-*/collection_manifest.json"))
manifest_rows = []
for path in manifest_paths:
    payload = read_json(path)
    manifest_rows.append({
        "shard": path.parent.name,
        "tasks": payload.get("n_tasks"),
        "rollouts": payload.get("n_rollouts"),
        "runs_per_temperature": payload.get("runs_per_temperature"),
        "total_samples_per_task": payload.get("total_samples_per_task"),
        "temperatures": payload.get("temperatures"),
        "workers": payload.get("rollout_workers"),
    })

manifests = pd.DataFrame(manifest_rows)
display(manifests if not manifests.empty else Markdown("_No collection manifests found._"))


In [ ]:
sample_rows = []
for path in sorted(run_root.glob("collection/shard-*/samples/sample-*/summary.json")):
    payload = read_json(path)
    sample_rows.append({
        "shard": path.parents[2].name,
        "sample": payload.get("sample_index"),
        "temperature": payload.get("temperature"),
        "temperature_run": payload.get("temperature_run_index"),
        "tasks": payload.get("n_tasks"),
        "completed": payload.get("n_completed"),
        "errors": payload.get("n_errors"),
        "with_patch": payload.get("n_with_patch"),
    })

sample_status = pd.DataFrame(sample_rows)
display(sample_status if not sample_status.empty else Markdown("_No sample summaries found._"))


## Prediction and evaluation overview

Merged predictions appear after the evaluation job starts. Until then, this cell falls back to the per-shard prediction files.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

prediction_rows = []
total_samples = int(manifests["total_samples_per_task"].max()) if not manifests.empty else 8
for sample_index in range(total_samples):
    merged = run_root / "merged" / f"sample-{sample_index}" / "predictions.jsonl"
    paths = [merged] if merged.is_file() else sorted(
        run_root.glob(f"collection/shard-*/samples/sample-{sample_index}/predictions.jsonl")
    )
    for path in paths:
        for row in read_jsonl(path):
            prediction_rows.append({
                **row,
                "sample_index": row.get("sample_index", sample_index),
                "patch_chars": len(row.get("model_patch", "")),
                "patch_present": bool(row.get("model_patch", "").strip()),
                "source": str(path),
            })

predictions = pd.DataFrame(prediction_rows)
if predictions.empty:
    display(Markdown("_No predictions found._"))
else:
    display(
        predictions.groupby(["sample_index", "temperature", "temperature_run_index"], dropna=False)
        .agg(trajectories=("instance_id", "size"), with_patch=("patch_present", "sum"), mean_patch_chars=("patch_chars", "mean"))
        .reset_index()
    )


In [ ]:
analysis_summary = read_json(run_root / "analysis" / "summary.json")
if analysis_summary:
    display(JSON(analysis_summary, expanded=False))
    rollouts_csv = run_root / "analysis" / "rollouts.csv"
    tasks_csv = run_root / "analysis" / "tasks.csv"
    rollout_analysis = pd.read_csv(rollouts_csv) if rollouts_csv.is_file() else pd.DataFrame()
    task_analysis = pd.read_csv(tasks_csv) if tasks_csv.is_file() else pd.DataFrame()
    display(task_analysis.head(20))
else:
    rollout_analysis = pd.DataFrame()
    task_analysis = pd.DataFrame()
    display(Markdown("_Analysis is not available yet. The dependent analysis job writes it after evaluation._"))


## Inspect one task and trajectory

Change `instance_id` and `sample_index` below. The cell shows the task, wrapper metadata, raw mini-swe-agent messages, generated patch, and evaluation report when each artifact exists.

In [ ]:
if predictions.empty:
    instance_id = ""
else:
    instance_id = str(predictions.iloc[0]["instance_id"])
sample_index = 0
print(f"instance_id={instance_id!r}, sample_index={sample_index}")


In [ ]:
trajectory_matches = sorted(
    run_root.glob(
        f"collection/shard-*/samples/sample-{sample_index}/trajectories/*/trajectory.json"
    )
)
trajectory_path = next(
    (path for path in trajectory_matches if read_json(path).get("instance_id") == instance_id),
    None,
)

if trajectory_path is None:
    display(Markdown("_Trajectory wrapper not found._"))
else:
    instance_dir = trajectory_path.parent
    task = read_json(instance_dir / "task.json")
    wrapper = read_json(trajectory_path)
    raw_paths = sorted(instance_dir.glob("**/*.traj.json"), key=lambda path: path.stat().st_mtime)
    raw = read_json(raw_paths[-1]) if raw_paths else {}
    display(Markdown(f"### Task: `{instance_id}` — sample {sample_index}"))
    if task:
        display(Markdown(task.get("problem_statement", "_No problem statement._")))
    display(JSON({key: wrapper.get(key) for key in ["status", "returncode", "patch_chars", "patch_source", "config"]}, expanded=False))
    messages = raw.get("messages", [])
    print(f"Raw messages: {len(messages)}")
    for index, message in enumerate(messages):
        role = message.get("role", "unknown")
        content = str(message.get("content", ""))
        display(Markdown(f"**{index}. {role}**\n\n```text\n{content[:6000]}\n```"))
    patch = wrapper.get("patch", "")
    display(Markdown(f"### Patch ({len(patch)} chars)\n\n```diff\n{patch[:12000]}\n```"))
    report_path = run_root / "evaluation" / f"sample-{sample_index}" / "logs" / instance_id / "report.json"
    if report_path.is_file():
        display(Markdown("### Evaluation report"))
        display(JSON(read_json(report_path), expanded=False))


## Triage incomplete runs

Use this table before scaling the pilot. It surfaces collection errors, empty patches, unevaluated rollouts, and evaluation failures.

In [ ]:
if rollout_analysis.empty:
    display(Markdown("_Run analysis first to populate triage data._"))
else:
    patch_present = rollout_analysis["patch_present"].astype(str).str.lower().eq("true")
    triage = rollout_analysis[
        (rollout_analysis["collection_status"].isin(["error", "missing"]))
        | (~patch_present)
        | (~rollout_analysis["evaluation_status"].isin(["resolved", "unresolved", "completed", "cached_report"]))
    ]
    display(triage[["instance_id", "sample_index", "temperature", "temperature_run_index", "collection_status", "patch_present", "evaluation_status", "trajectory_path"]].head(100))
